# Regime Shift Recovery, Using Only Clean Shifts

Our first attempt found 26 "shift points", but many were clustered close
together (e.g. windows 9, 10, 12, 13), which looks like noise from the
median-split regime definition flipping back and forth, not real
transitions between stable periods.

Here we require a regime to hold for at least 8 windows in a row before
we count it as a real period, then only look at shifts between these
clean, stable periods.

In [1]:
import numpy as np

In [2]:
forecasts = np.load("../data/forecasts.npy")
actuals = np.load("../data/actuals.npy")
regimes = np.load("../data/sensex_regime_labels.npy", allow_pickle=True)

day = 15
day_forecasts = forecasts[:, day, :]
day_actuals = actuals[:, day]

lower = day_forecasts[:, 1]
upper = day_forecasts[:, 9]

n = len(day_actuals)
split_point = int(n * 0.6)

cal_lower, test_lower = lower[:split_point], lower[split_point:]
cal_upper, test_upper = upper[:split_point], upper[split_point:]
cal_actuals, test_actuals = day_actuals[:split_point], day_actuals[split_point:]
test_regimes = regimes[split_point:]

len(test_actuals)

141

## Recompute fixed and adaptive coverage, point by point (same as before)

In [10]:
cal_scores = np.maximum(cal_lower - cal_actuals, cal_actuals - cal_upper)
q_hat = np.quantile(cal_scores, 0.8)

fixed_lower = test_lower - q_hat
fixed_upper = test_upper + q_hat
fixed_covered = ((test_actuals >= fixed_lower) & (test_actuals <= fixed_upper)).astype(int)

target_alpha = 0.2
gamma = 0.01
alpha_t = target_alpha
score_history = list(cal_scores)
adaptive_covered = []

for i in range(len(test_actuals)):
    alpha_t = np.clip(alpha_t, 0.01, 0.5)
    q_t = np.quantile(score_history, 1 - alpha_t)
    lo = test_lower[i] - q_t
    hi = test_upper[i] + q_t
    actual = test_actuals[i]
    covered = (actual >= lo) and (actual <= hi)
    adaptive_covered.append(int(covered))
    err_t = 0 if covered else 1
    alpha_t = alpha_t + gamma * (target_alpha - err_t)
    this_score = max(test_lower[i] - actual, actual - test_upper[i])
    score_history.append(this_score)

adaptive_covered = np.array(adaptive_covered)

## Find stable regime runs

A "run" is a stretch of consecutive windows with the same regime label.
We only keep runs that last at least 8 windows, treating shorter runs as
noise from the median-split boundary rather than a real period.

In [11]:
min_run_length = 8

runs = []
run_start = 0
current_regime = test_regimes[0]

for i in range(1, len(test_regimes)):
    if test_regimes[i] != current_regime:
        runs.append((current_regime, run_start, i))  # regime, start, end (exclusive)
        run_start = i
        current_regime = test_regimes[i]
runs.append((current_regime, run_start, len(test_regimes)))

# keep only long enough runs
clean_runs = [r for r in runs if (r[2] - r[1]) >= min_run_length]

print("total runs found:", len(runs))
print("clean runs (length >=", min_run_length, "):", len(clean_runs))
for r in clean_runs:
    print(r, "length:", r[2] - r[1])

total runs found: 27
clean runs (length >= 8 ): 6
(np.str_('high_vol'), 0, 9) length: 9
(np.str_('high_vol'), 13, 21) length: 8
(np.str_('low_vol'), 21, 33) length: 12
(np.str_('high_vol'), 33, 49) length: 16
(np.str_('low_vol'), 62, 87) length: 25
(np.str_('low_vol'), 121, 137) length: 16


## Clean shift points

A clean shift is the start of each clean run, except the very first one
(since there's nothing before it to shift from).

In [12]:
# only count it as a shift if the regime actually changed
# from the previous kept run (not just because a short noisy run
# in between got filtered out)
clean_shift_points = []
for i in range(1, len(clean_runs)):
    prev_regime = clean_runs[i-1][0]
    this_regime = clean_runs[i][0]
    if prev_regime != this_regime:
        clean_shift_points.append(clean_runs[i][1])

print("clean shift points (real regime changes only):", clean_shift_points)

clean shift points (real regime changes only): [21, 33, 62]


## Compare recovery: fixed vs adaptive, using only clean shifts

In [13]:
recovery_window = 5

fixed_recovery_scores = []
adaptive_recovery_scores = []

for sp in clean_shift_points:
    end = min(sp + recovery_window, len(test_actuals))
    fixed_recovery_scores.append(fixed_covered[sp:end].mean())
    adaptive_recovery_scores.append(adaptive_covered[sp:end].mean())

if len(clean_shift_points) > 0:
    print("fixed conformal coverage right after a clean shift:", round(np.mean(fixed_recovery_scores) * 100, 1), "%")
    print("adaptive conformal coverage right after a clean shift:", round(np.mean(adaptive_recovery_scores) * 100, 1), "%")
else:
    print("no clean shifts found with this minimum run length, try lowering min_run_length")

fixed conformal coverage right after a clean shift: 86.7 %
adaptive conformal coverage right after a clean shift: 86.7 %


## Baseline: overall coverage for comparison

In [14]:
print("fixed conformal overall coverage:", round(fixed_covered.mean() * 100, 1), "%")
print("adaptive conformal overall coverage:", round(adaptive_covered.mean() * 100, 1), "%")

fixed conformal overall coverage: 78.0 %
adaptive conformal overall coverage: 78.7 %


**Regime shift recovery speed (inconclusive):** we tested whether adaptive
conformal prediction recovers coverage faster than fixed conformal prediction
immediately following a genuine volatility regime transition. After filtering
out short, noisy regime flips (requiring a minimum stable run of 8 windows),
only 3 genuine regime transitions were identified within the test set. Both
methods showed identical coverage (86.7%) in the 5 windows following these
transitions. Given the small number of genuine shifts available in this
sample, this result should be treated as inconclusive rather than evidence
of no difference: a dataset spanning a longer time period, with more regime
transitions, would be needed to properly test recovery speed. This is noted
as a direction for future work rather than a settled finding.

## **RELATED WORK**

## Related Work

**Conformal prediction fundamentals.** Split conformal prediction provides
distribution-free, finite-sample coverage guarantees under an exchangeability
assumption (Angelopoulos & Bates, 2021). Gibbs & Candes (2021) relax this
assumption for non-stationary settings via adaptive conformal inference,
which updates the target miscoverage rate online based on recent coverage
performance -- the approach we build on in this work.

**Conformal prediction for financial time series.** Regime-awareness in
financial conformal calibration is an active area. Recent work has proposed
regime-weighted conformal calibration for nonstationary Value-at-Risk
estimation, Mondrian conformal variants that partition forecasts into
high- and low-volatility regimes to reveal coverage imbalances, and
Temporal Conformal Prediction, an online framework combining quantile
forecasting with real-time conformal recalibration for financial risk
estimation. These works establish that volatility-regime-conditional
calibration is both a real phenomenon and a tractable target for correction
in financial markets, but do not evaluate time series foundation models
specifically.

**Time series foundation models.** Chronos and related zero-shot foundation
models have been evaluated for financial forecasting accuracy: one study
compared zero-shot and fine-tuned Chronos against classical and deep
learning baselines on Nasdaq-100 and S&P 500 data, finding zero-shot Chronos
statistically comparable to top traditional methods on accuracy, though
weaker on simulated trading performance. Separately, finance-specific
foundation models such as Kronos have been proposed, pretrained directly on
financial candlestick data rather than general time series. Existing
evaluations of Chronos-2's calibration specifically (as opposed to point
accuracy) have focused on stable domains such as energy grid operations,
where the model was found to be slightly conservative under a fixed
nominal coverage level.

**Gap addressed by this work.** To our knowledge, no existing study
examines whether a general-purpose (non-finance-pretrained) time series
foundation model's calibration is conditional on volatility regime,
specifically in the context of Indian equity markets (Sensex, Nifty),
which combine emerging-market volatility characteristics with limited
prior TSFM evaluation. This work addresses that gap by combining
horizon-stratified calibration diagnostics, regime-conditional coverage
analysis, and both fixed and adaptive conformal correction.